# Baseline B3

Author: Mathias Marryat Husted

This notebook provides a reproducible way to run the baseline B3 runs. Checks are performed at each step to ensure data consistency.

Baseline B3 focuses on direct API matching.

In [1]:
import requests
print(requests.get("https://rest.genenames.org/info", headers={"Accept": "application/json"}).json()["lastModified"])

2026-10-27T13:01:56.131Z


API access date: 2026-10-07

HGNC last modified on 2026-10-06

In [7]:
import pandas as pd
from urllib.parse import quote
import sys
sys.path.append("../shared")
from b3_resolver import load_cache, handle_api_lookup
from scoring import decide

# Reload the benchmark files

BENCHMARK_PATH = "../../data/dataset/benchmark.csv"
CONTROL_PATH = "../../data/dataset/control.csv"
FIELD_ORDER = ["symbol", "prev_symbol", "alias_symbol", "name", "prev_name", "alias_name"] # Ruleset fallback order
CACHE_PATH = "../../data/cache/api.jsonl"

benchmark = pd.read_csv(BENCHMARK_PATH, dtype=str, keep_default_na=False)
control = pd.read_csv(CONTROL_PATH, dtype=str, keep_default_na=False)


In [3]:
# Load the cache file into memory
cache = load_cache(CACHE_PATH)

In [4]:
# Perform the API matching against the benchmark and create the table

def run_b3(data):
    api_lookup_result = []
    for row_id, name, entity_type in zip(data["id"], data["input"], data["entity_type"]):
        print(f"Looking up {row_id}")
        result = handle_api_lookup(name, entity_type, cache, CACHE_PATH)
        result["id"] = row_id
        result["input"] = name
        result["entity_type"] = entity_type

        api_lookup_result.append(result)
    return pd.DataFrame(api_lookup_result)

b3 = run_b3(benchmark)
b3.to_csv("../../results/baselines/b3.csv", index=False)

Looking up CHEM-001
Looking up CHEM-002
Looking up CHEM-003
Looking up CHEM-004
Looking up CHEM-005
Looking up CHEM-006
Looking up CHEM-007
Looking up CHEM-008
Looking up CHEM-009
Looking up CHEM-010
Looking up CHEM-011
Looking up CHEM-012
Looking up CHEM-013
Looking up CHEM-014
Looking up CHEM-015
Looking up CHEM-016
Looking up CHEM-017
Looking up CHEM-018
Looking up CHEM-019
Looking up CHEM-020
Looking up CHEM-021
Looking up CHEM-022
Looking up CHEM-023
Looking up CHEM-024
Looking up CHEM-025
Looking up CHEM-026
Looking up CHEM-027
Looking up CHEM-028
Looking up CHEM-029
Looking up CHEM-030
Looking up CHEM-031
Looking up CHEM-032
Looking up CHEM-033
Looking up CHEM-034
Looking up CHEM-035
Looking up CHEM-036
Looking up CHEM-037
Looking up CHEM-038
Looking up CHEM-039
Looking up CHEM-040
Looking up CHEM-041
Looking up CHEM-042
Looking up CHEM-043
Looking up CHEM-044
Looking up CHEM-045
Looking up CHEM-046
Looking up CHEM-047
Looking up CHEM-048
Looking up CHEM-049
Looking up CHEM-050


Verify achievable ceiling (check how many control.csv entities actually correctly resolve using the b3_resolver).

In [5]:
b3_control = run_b3(control)
b3_control.to_csv("../../results/baselines/b3_control.csv", index=False)

Looking up CTRL-001
Looking up CTRL-002
Looking up CTRL-003
Looking up CTRL-004
Looking up CTRL-005
Looking up CTRL-006
Looking up CTRL-007
Looking up CTRL-008
Looking up CTRL-009
Looking up CTRL-010
Looking up CTRL-011
Looking up CTRL-012
Looking up CTRL-013
Looking up CTRL-014
Looking up CTRL-015
Looking up CTRL-016
Looking up CTRL-017
Looking up CTRL-018
Looking up CTRL-019
Looking up CTRL-020
Looking up CTRL-021
Looking up CTRL-022
Looking up CTRL-023
Looking up CTRL-024
Looking up CTRL-025
Looking up CTRL-026
Looking up CTRL-027
Looking up CTRL-028
Looking up CTRL-029
Looking up CTRL-030
Looking up CTRL-031
Looking up CTRL-032
Looking up CTRL-033
Looking up CTRL-034
Looking up CTRL-035
Looking up CTRL-036
Looking up CTRL-037
Looking up CTRL-038
Looking up CTRL-039
Looking up CTRL-040
Looking up CTRL-041
Looking up CTRL-042
Looking up CTRL-043
Looking up CTRL-044
Looking up CTRL-045
Looking up CTRL-046
Looking up CTRL-047
Looking up CTRL-048
Looking up CTRL-049
Looking up CTRL-050


In [ ]:
control = pd.read_csv("../../data/dataset/control.csv", dtype=str, keep_default_na=False)
b3_control = pd.read_csv("../../results/baselines/b3_control.csv", dtype=str, keep_default_na=False)
ctrl = pd.merge(b3_control, control[["id", "id_status", "accepted_id"]], on="id", validate="one_to_one")
ctrl["n_candidates"] = ctrl["n_candidates"].astype(int)
ctrl["outcome"] = ctrl.apply(decide, axis=1)
pd.crosstab(ctrl["entity_type"], ctrl["outcome"])

outcome,abstain_multiple,abstained_on_none,correct
entity_type,,,
chemical,2,1,106
gene,0,0,80


In [9]:
ctrl[ctrl["outcome"] != "correct"][["id", "input", "n_candidates", "candidate_ids", "accepted_id", "id_status"]]

,id,input,n_candidates,candidate_ids,accepted_id,id_status
34,CTRL-035,D-glucose 6-phosphate,2,439958;99058,439958;99058,ambiguous
84,CTRL-085,NAD+,4,5892;5893;15938971;10897651,5892,single
169,CTRL-170,phosphatidylcholine,0,,,none


3 chemicals could not be resolved via the retrieval layer, even using their gold identifiers.